# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

* Load the packages and the MMLU datasets

In [14]:

#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import general_utils
#Import other answer generation functions as needed
from evaluate_model_fast import batch_eval
from rag import rag_pipeline, load_vector_store
from summarize_accuracy import summarize_accuracy
import os

# load Gemini client
client = general_utils.load_gemini_client()

# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = range(1,9,1)

# Range of few shot examples to include in prompt
few_shot_range = range(1,6,1)

# round number
round_num = 5


## Evaluation of Gemini's performance on MMLU datasets

1. Evaluate Standard Model Performance

We simply prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is very basic prompt with no prompt engineering or chain-of-thought prompting.

2. Evaluate Chain-of-Thought (COT) Model Performance

We prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is asking the model to think step-by-step before answering the question. We created various prompt engineering techniques to see which prompt engineering techniques are most effective for Gemini. The techniques are:

* Basic COT asking the model to think step-by-step
* COT asking the model to take its time to think as much as possible
* COT with emotional manipulation (e.g., I will give you $1000 if you get the question right)
* COT hinting that I'm from a medical career (previous work shows model tries extra hard when it thinks the context is for human medical field)
* COT with persona (e.g., you are an expert in {domain} with 50 years of working experience)
* COT with few shot examples. We provided 1, 2, ..., 5 examples of questions and correct answers to the model to test how does the number of examples affect the performance of the model.

3. Evaluate Model Performance with RAG

We converted concept net to RAG vector store and used it as context to answer the questions from the MMLU datasets. We tested different k values for the RAG model, k in the range of 1 to 8 (inclusive) with step size 1. We also tested 2 variants for the RAG prompt engineering techniques: 

* RAG asking the model to only use the provided content to answer the question
* RAG asking the model to use the provided content to answer the question if it is helpful and relevant, but have the option to disregard the content if it is not helpful or not relevant.

> 💡 Important: I've created the vector store, saved it locally in a folder called "chroma_langchain_db", zipped the folder and uploaded to this [Google Drive link](https://drive.google.com/file/d/12_Yagf9kUyb3b_0hGDyRVrop_ls9lRTg/view?usp=sharing). You can download the file, unzip it and place it under `scripts/chroma_langchain_db`, then continue running the code below to load the vector store locally.

In [5]:
# reload cached vector store. Must be saved under "scripts/chroma_langchain_db" if from the root directory
vector_store = load_vector_store(persist_directory="./chroma_langchain_db")

> ⚠️ Warning: Takes ~50 minutes if you want to create the vector store from scratch by running the code below.

In [4]:
# build the rag pipeline to vectorize concept net
#file_path = "../input_data/concept_net.txt"
#vector_store = rag_pipeline(file_path, split_by="newline", chunk_size=1000, chunk_overlap=0, add_start_index=True, batch_processing_size=700)

Run all the variants in batch below: 
> Note that I've run this manually for round 1-5.

In [12]:
master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=False,
    vector_store=vector_store,
    max_workers=10,
    folder_path=f"../results/round{round_num}",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:   0%|          | 0/30 [00:00<?, ?it/s]

Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)


Batch Evaluation Progress:   3%|▎         | 1/30 [02:24<1:09:53, 144.61s/it]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 4 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 5 failed: Unterminated string starting at: line 2 column 16 (char 17)


Batch Evaluation Progress:   7%|▋         | 2/30 [06:34<1:36:19, 206.43s/it]

...

Batch Evaluation Progress:  43%|████▎     | 13/30 [36:02<47:54, 169.06s/it]

Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 2 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 4 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 5 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 6 failed: Unterminated string starting at: line 2 column

Batch Evaluation Progress:  47%|████▋     | 14/30 [39:54<50:09, 188.12s/it]

Attempt 10 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 4 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 colum

Batch Evaluation Progress: 100%|██████████| 30/30 [52:53<00:00, 105.80s/it]


In [13]:

# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }


# Range of k values to try for RAG
k_range = range(2,8,1)

# Range of few shot examples to include in prompt
few_shot_range = range(2,5,1)

# round number
round_num = 5

master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=False,
    vector_store=vector_store,
    max_workers=10,
    folder_path=f"../results/round{round_num}",
    few_shot_range=few_shot_range)


Batch Evaluation Progress:   7%|▋         | 3/45 [01:57<27:43, 39.61s/it]

Attempt 1 failed: Unterminated string starting at: line 2 column 15 (char 16)


Batch Evaluation Progress:  13%|█▎        | 6/45 [04:37<32:40, 50.27s/it]

Attempt 1 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 2 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 3 failed: Unterminated string starting at: line 2 column 15 (char 16)


Batch Evaluation Progress:  27%|██▋       | 12/45 [11:23<31:26, 57.16s/it]

...

Batch Evaluation Progress:  62%|██████▏   | 28/45 [55:56<43:44, 154.39s/it]

Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column 16 (char 17)


Batch Evaluation Progress:  64%|██████▍   | 29/45 [57:53<38:13, 143.34s/it]

Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 2 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 1 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 15 (char 16)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 3 failed: Unterminated string starting at: line 2 column

Batch Evaluation Progress: 100%|██████████| 45/45 [1:11:27<00:00, 95.28s/it] 


## Average accuracy comparison among all variants

In [52]:
#create a dataframe to summarize the results
master_results_df = pd.read_csv(f'../results/round{round_num}/master_results_df.csv')
df_accuracy_by_prompt_subject = summarize_accuracy(master_results_df, output_file=f'../results/round{round_num}/accuracy_by_prompt_subject.csv', group_by=['prompt_type', 'subject'])

Filtered out 6 rows where the model output was an 'Error'
subject                     anatomy  formal_logic  moral_scenarios
prompt_type                                                       
cot                         0.91000      0.980000          0.89000
cot_emotional_manipulation  0.91000      0.990000          0.87000
cot_few_shot                0.91000      0.988000          0.84400
cot_max_thinking            0.92000      0.980000          0.81000
cot_medical_career          0.91000      0.990000          0.85000
cot_persona                 0.89000      0.980000          0.87000
rag                         0.80875      0.979899          0.85250
rag_optional                0.79625      0.982456          0.84625
standard                    0.90000      0.970000          0.87000


## Compare accuracy between 0-shot and few-shot learning

In [53]:
# few shot
few_shot_df = master_results_df.copy()
few_shot_df['shot_n'] = few_shot_df['shot_n'].fillna(0).astype(int)
type_of_interest = ['cot', 'cot_few_shot']
few_shot_df = few_shot_df[few_shot_df['prompt_type'].isin(type_of_interest)]
accuracy_by_few_shot_n = summarize_accuracy(few_shot_df, output_file=f'../results/round{round_num}/accuracy_by_few_shot_n.csv', group_by=['shot_n', 'subject'], unstack_by='subject')

Filtered out 0 rows where the model output was an 'Error'
subject  anatomy  formal_logic  moral_scenarios
shot_n                                         
0           0.91          0.98             0.89
1           0.91          0.99             0.85
2           0.89          0.98             0.81
3           0.92          0.99             0.86
4           0.91          0.99             0.84
5           0.92          0.99             0.86


## Compare the effect of the value of k in RAG on the accuracy of the model

In [54]:
# RAG effect of K: 'rag'
rag_df = master_results_df.copy()
type_of_interest = ['rag']
rag_df = rag_df[rag_df['prompt_type'].isin(type_of_interest)]
rag_df['k'] = rag_df['k'].astype(int)
accuracy_by_rag_k = summarize_accuracy(rag_df, output_file=f'../results/round{round_num}/accuracy_by_rag_k.csv', group_by=['k', 'subject'], unstack_by='subject')

Filtered out 4 rows where the model output was an 'Error'
subject  anatomy  formal_logic  moral_scenarios
k                                              
1           0.90      0.969697             0.83
2           0.82      0.980000             0.86
3           0.80      0.979798             0.83
4           0.77      0.969697             0.87
5           0.79      0.980000             0.86
6           0.79      0.989899             0.85
7           0.82      0.980000             0.86
8           0.78      0.990000             0.86


In [55]:
# RAG effect of K: 'rag_optional'
rag_df = master_results_df.copy()
type_of_interest = ['rag_optional']
rag_df = rag_df[rag_df['prompt_type'].isin(type_of_interest)]
rag_df['k'] = rag_df['k'].astype(int)
accuracy_by_rag_k = summarize_accuracy(rag_df, output_file=f'../results/round{round_num}/accuracy_by_rag_optional_k.csv', group_by=['k', 'subject'], unstack_by='subject')

Filtered out 2 rows where the model output was an 'Error'
subject  anatomy  formal_logic  moral_scenarios
k                                              
1           0.85      0.979592             0.84
2           0.82      0.980000             0.85
3           0.78      0.970000             0.83
4           0.81      0.990000             0.87
5           0.77      0.990000             0.84
6           0.78      0.990000             0.88
7           0.76      0.970000             0.80
8           0.80      0.990000             0.86


In [56]:
# Filter for all errors first
all_errors = master_results_df[master_results_df['correct'] == False]

# Group by prompt and sample 20 from each group
error_samples_df = all_errors.groupby(['prompt_type', 'subject', 'shot_n', 'k']).apply(
    lambda x: x.sample(n=min(20, len(x)), random_state=42)
).reset_index()

error_samples_df = error_samples_df.sort_values(by=['prompt_type', 'subject', 'shot_n', 'k'])

error_samples_df.to_csv(f'../results/round{round_num}/model_error_samples.csv', index=False)

## Intra-model consistency among different runs on the same dataset